### **Setup & Import Libraries**

In [10]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

# Import RAGAS metrics
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision

# Import LangChain components
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

# Import RAG components
from rag.chain import RAGChain
from rag.core.retrieval.retriever import RAGRetriever
from rag.core.generation.generator import RAGGenerator

# Import config
from app.config import settings

C:\Users\Ratuayu Nurfajar\AppData\Local\Temp\ipykernel_9432\2699610101.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision
C:\Users\Ratuayu Nurfajar\AppData\Local\Temp\ipykernel_9432\2699610101.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision


### **Load Test Data**

In [2]:
# Load test data
test_data_path = Path('test_data.json')

if not test_data_path.exists():
    print(f"Error: {test_data_path} not found")
else:
    with open(test_data_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    print(f"Loaded {len(test_data)} test cases")

Loaded 15 test cases


### **Initialize RAGAS Metrics**

In [3]:
try:
    eval_llm = ChatOpenAI(
        model=settings.MODEL_NAME,
        openai_api_key=settings.OPENAI_API_KEY,
        openai_api_base=settings.OPENAI_BASE_URL,
        temperature=0.0
    )
    
    eval_embeddings = HuggingFaceEmbeddings(
        model_name=settings.EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'}
    )
    
    faithfulness.llm = eval_llm
    context_precision.llm = eval_llm
    context_precision.embeddings = eval_embeddings
    
    print("RAGAS metrics initialized!")
    print(f"   LLM Model: {settings.MODEL_NAME}")
    print(f"   Embedding Model: {settings.EMBEDDING_MODEL}")
    print(f"   Base URL: {settings.OPENAI_BASE_URL}")
    
except Exception as e:
    print(f"Error initializing RAGAS: {str(e)}")
    print("Make sure .env file has OPENAI_API_KEY and OPENAI_BASE_URL")
    raise

C:\Users\Ratuayu Nurfajar\AppData\Local\Temp\ipykernel_9432\2608459111.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  eval_embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1013.54it/s, Materializing param=pooler.dense.weight]                               


RAGAS metrics initialized!
   LLM Model: openai/gpt-4o-mini
   Embedding Model: BAAI/bge-m3
   Base URL: https://openrouter.ai/api/v1


### **Generate Predictions for Evaluation**

In [12]:
# Initialize RAG components
retriever = RAGRetriever()
generator = RAGGenerator()

evaluation_data = []

print("Generating predictions...\n")

for idx, test_case in enumerate(test_data, 1):
    question = test_case['question']
    ground_truth = test_case['ground_truth']
    provided_contexts = test_case['contexts']
    
    try:
        # Generate answer
        context = "\n\n".join(provided_contexts)
        result = generator.generate(question=question, context=context)
        generated_answer = result.get('content', '')
        
        evaluation_data.append({
            "question": question,
            "contexts": provided_contexts,
            "response": generated_answer,
            "ground_truth": ground_truth,
        })
        
        print(f"✓ [{idx}/{len(test_data)}] Generated: {question[:50]}...")
        
    except Exception as e:
        print(f"✗ [{idx}/{len(test_data)}] Error: {str(e)[:80]}")

print(f"\nGenerated {len(evaluation_data)} predictions")

Proses Retriever


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 290.18it/s, Materializing param=pooler.dense.weight]                               
d:\document\coding\project\Spacer Chat\.venv\Lib\site-packages\transformers\tokenization_utils_tokenizers.py:128: RuntimeWarning: coroutine 'calculate_faithfulness' was never awaited
  vocab = [tuple(item) for item in vocab]


Proses Generator
Generating predictions...

✓ [1/15] Generated: Apa yang dimaksud dengan machine learning dalam ko...
✓ [2/15] Generated: Sebutkan dua jenis utama metode learning pada data...
✓ [3/15] Generated: Apa yang dimaksud dengan metode klasifikasi dalam ...
✓ [4/15] Generated: Bagaimana ide dasar dari metode regresi dalam supe...
✓ [5/15] Generated: Jelaskan apa fungsi dari metode clustering pada un...
✓ [6/15] Generated: Apa definisi dari Business Intelligence (BI) menur...
✓ [7/15] Generated: Sebutkan 4 komponen utama yang menyusun Arsitektur...
✓ [8/15] Generated: Sebutkan dan jelaskan 3 tipe Business Analytics be...
✓ [9/15] Generated: Apa yang dimaksud dengan Dashboard dalam ekosistem...
✓ [10/15] Generated: Bagaimana penerapan Business Intelligence dapat me...
✓ [11/15] Generated: Jelaskan apa yang dimaksud dengan model Client-Ser...
✓ [12/15] Generated: Sebutkan perbedaan antara data dinamis dan data st...
✓ [13/15] Generated: Apa fungsi utama dari pemrograman client sid

### **Evaluation**

In [13]:
dataset = Dataset.from_list(evaluation_data)

print("Running evaluation...\n")

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        context_precision
    ]
)

df = result.to_pandas()


Running evaluation...



Evaluating: 100%|██████████| 30/30 [00:59<00:00,  1.99s/it]


In [16]:
print("\n=========== Evaluation Summary ===========\n")

print(f"Faithfulness Mean: {df['faithfulness'].mean():.4f}")
print(f"Context Precision Mean: {df['context_precision'].mean():.4f}")


=========== Evaluation Summary ===========

Faithfulness Mean: 0.7109
Context Precision Mean: 0.8944
